# Train the augmented CIFAR-10 CNN with a GPU

To use a Colab GPU while keeping the notebook open in VS Code, connect this notebook to a **Colab remote runtime** with the installed Colab extension, then choose a GPU runtime. The code executes on Colab's cloud machine, not on this Mac; a local VS Code kernel cannot borrow Colab's GPU. This repository is public, so the first code cell clones it without a token. The next cell installs dependencies and downloads CIFAR-10, and the final cell starts training on CUDA. A local Mac kernel would need Apple MPS instead; this project previously hit an MPS backend crash with its installed PyTorch version.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Chiebukar/cifar10-image-classifier.git"
PROJECT_DIR = Path("/content/cifar10-image-classifier")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").exists():
        raise FileExistsError(
            f"{PROJECT_DIR} exists but is not a Git checkout. "
            "Remove or rename it, then rerun this cell."
        )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

print(f"Project ready at {PROJECT_DIR}")

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
import subprocess
import sys

# Install project dependencies before importing the training modules.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

# Download CIFAR-10 into the cloned project's data/raw directory.
subprocess.run(
    [sys.executable, "-m", "src.data.download_dataset"],
    cwd=PROJECT_DIR,
    check=True,
)

# Recreate the deterministic split only if it is absent from the checkout.
split_file = PROJECT_DIR / "configs" / "data_splits.json"
if not split_file.exists():
    subprocess.run(
        [sys.executable, "-m", "src.data.create_splits"],
        cwd=PROJECT_DIR,
        check=True,
    )

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is active. In Colab, choose Runtime > Change runtime type > GPU, "
        "then rerun the notebook cells."
    )

training_env = os.environ.copy()
training_env["CIFAR10_DEVICE"] = "cuda"
subprocess.run(
    [sys.executable, "-u", "-m", "src.training.train_custom_cnn_augmented"],
    cwd=PROJECT_DIR,
    env=training_env,
    check=True,
)